# 🧲 03 · 双塔召回与向量检索

> 工业推荐/搜索的召回主力：**双塔（DSSM）**——用户塔和物品塔分别产出 embedding，点积/余弦衡量相关性。
> 本节手写双塔前向与训练（含负采样），并用向量检索 demo 展示召回 top-k。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 双塔为什么是召回标配？ | 离线算物品向量、在线只算用户向量 |
| 2 | 怎么训练？ | 内积 + softmax/负采样损失 |
| 3 | 负样本怎么选？ | 随机负采样 / 难负样本 |
| 4 | 线上怎么快速检索？ | ANN 向量索引（HNSW/IVF） |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 构造用户/物品特征

In [2]:
n_u, n_i = 150, 300
feat_dim = 16
u_feat = rng.normal(0, 1, (n_u, feat_dim))   # 用户特征（年龄档/偏好/历史统计）
i_feat = rng.normal(0, 1, (n_i, feat_dim))   # 物品特征（类目/价格档/内容向量）
print('用户特征:', u_feat.shape, '物品特征:', i_feat.shape)
print('➡️ 真实场景中塔输入 = 稀疏 id + 稠密特征的 embedding 拼接')

用户特征: (150, 16) 物品特征: (300, 16)
➡️ 真实场景中塔输入 = 稀疏 id + 稠密特征的 embedding 拼接


## 2️⃣ 手写双塔：两塔 MLP → 点积

In [3]:
def tower_forward(x, W1, b1, W2, b2):
    h = np.maximum(0, x @ W1 + b1)
    return h @ W2 + b2

# 两塔各 2 层（输出 8 维）
emb_dim = 8
W1u = rng.normal(0, 0.1, (feat_dim, 16)); b1u = np.zeros(16)
W2u = rng.normal(0, 0.1, (16, emb_dim)); b2u = np.zeros(emb_dim)
W1i = rng.normal(0, 0.1, (feat_dim, 16)); b1i = np.zeros(16)
W2i = rng.normal(0, 0.1, (16, emb_dim)); b2i = np.zeros(emb_dim)

u_emb = tower_forward(u_feat, W1u, b1u, W2u, b2u)
i_emb = tower_forward(i_feat, W1i, b1i, W2i, b2i)
print('用户塔输出:', u_emb.shape, '物品塔输出:', i_emb.shape)
sim = u_emb @ i_emb.T   # (n_u, n_i)
print('相似度矩阵:', sim.shape)
assert sim.shape == (n_u, n_i)
print('✅ 双塔前向完成')

用户塔输出: (150, 8) 物品塔输出: (300, 8)
相似度矩阵: (150, 300)
✅ 双塔前向完成


## 3️⃣ 构造训练样本：正样本 + 随机负采样

In [4]:
# 正样本：每用户 5 个"感兴趣"物品；负样本：每用户 60 个随机采样
pos_set = np.array([rng.choice(n_i, size=5, replace=False) for _ in range(n_u)])
neg_set = np.array([rng.choice(n_i, size=60, replace=False) for _ in range(n_u)])
print(f'每个用户正样本 {pos_set.shape[1]} 个、负样本 {neg_set.shape[1]} 个')
print('➡️ 工业上还会用「难负样本」（曝光未点击）提升训练质量')

每个用户正样本 5 个、负样本 60 个
➡️ 工业上还会用「难负样本」（曝光未点击）提升训练质量


## 4️⃣ 训练：内积相似度 + 交叉熵（手写梯度）

In [5]:
# 简易双塔训练：直接用用户/物品 embedding + 内积 + 交叉熵（向量化批量更新）
emb_dim = 8
lr2, epochs2 = 0.1, 100
U_emb = rng.normal(0, 0.1, (n_u, emb_dim))
I_emb = rng.normal(0, 0.1, (n_i, emb_dim))

loss_hist = []
for ep in range(epochs2):
    tot = 0.0
    for u in range(n_u):
        pidx, nidx = pos_set[u], neg_set[u]
        s_p = U_emb[u] @ I_emb[pidx].T          # (5,)
        s_n = U_emb[u] @ I_emb[nidx].T          # (60,)
        p_p = 1/(1+np.exp(-s_p)); p_n = 1/(1+np.exp(-s_n))
        e_p = p_p - 1.0; e_n = p_n - 0.0
        # 批量更新
        U_emb[u] -= lr2 * (e_p @ I_emb[pidx] + e_n @ I_emb[nidx]) / 65
        I_emb[pidx] -= lr2 * e_p[:, None] * U_emb[u][None, :]
        I_emb[nidx] -= lr2 * e_n[:, None] * U_emb[u][None, :]
        # 标准交叉熵（正负样本各一半权重）
        loss = (-np.log(p_p + 1e-9).mean() - np.log(1 - p_n + 1e-9).mean()) / 2
        tot += loss
    loss_hist.append(tot / n_u)
    if ep % 25 == 0:
        print(f'epoch {ep}: loss = {loss_hist[-1]:.4f}')

plt.figure(figsize=(6, 3))
plt.plot(loss_hist, color='#2563eb')
plt.xlabel('epoch'); plt.ylabel('平均 -log 概率'); plt.title('双塔训练损失')
plt.show()
assert loss_hist[-1] < 0.5
print('✅ 训练收敛')

epoch 0: loss = 0.6935


epoch 25: loss = 1.0300


epoch 50: loss = 0.8976


epoch 75: loss = 0.6280


✅ 训练收敛


C:\Users\24260\AppData\Local\Temp\ipykernel_40712\3991839886.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5️⃣ 召回 demo：给用户找 top-k 物品

In [6]:
score = U_emb @ I_emb.T
# 召回：给几个用户看 top-k
for u in [0, 5, 42]:
    top = np.argsort(-score[u])[:10]
    print(f'用户{u+1} 召回 top-5 物品: {[int(i)+1 for i in top[:5]]}')

# 评估：正样本被召回进 top-20 的比例
hit = 0
for u in range(n_u):
    top20 = np.argsort(-score[u])[:20]
    hit += np.isin(pos_set[u], top20).sum()
recall20 = hit / (n_u * 5)
print(f'\n召回@20 对正样本的命中率: {recall20:.2%}')
assert recall20 > 0.5
print('✅ 双塔学会把正样本排前面（远高于随机基线 ≈33%）')

用户1 召回 top-5 物品: [272, 33, 136, 95, 70]
用户6 召回 top-5 物品: [239, 109, 39, 186, 190]
用户43 召回 top-5 物品: [195, 33, 286, 200, 127]

召回@20 对正样本的命中率: 73.47%
✅ 双塔学会把正样本排前面（远高于随机基线 ≈33%）


## 6️⃣ 向量检索：暴力搜索 vs ANN 思想

In [7]:
import time
# 暴力检索（精确）：用户向量 × 全量物品向量
t0 = time.time()
for u in range(n_u):
    _ = u_emb[u] @ i_emb.T
brute_t = time.time() - t0

print(f'暴力检索耗时（{n_i} 物品）: {brute_t*1000:.1f} ms')
print('''➡️ 线上物品量级千万~亿级，暴力不行，用 ANN 索引：
   - HNSW：分层小世界图，召回率/时延平衡；
   - IVF：倒排聚类，先粗筛再细算；
   - PQ：乘积量化压缩向量，省内存。
   指标：召回率@K vs 时延 vs 内存。''')

暴力检索耗时（300 物品）: 1.3 ms
➡️ 线上物品量级千万~亿级，暴力不行，用 ANN 索引：
   - HNSW：分层小世界图，召回率/时延平衡；
   - IVF：倒排聚类，先粗筛再细算；
   - PQ：乘积量化压缩向量，省内存。
   指标：召回率@K vs 时延 vs 内存。


## 📝 自测清单

- [ ] 能画出双塔结构并解释为什么适合召回（离线/在线解耦）
- [ ] 能手写双塔前向与负采样训练
- [ ] 能解释随机负采样与难负样本的区别
- [ ] 能说出 HNSW/IVF/PQ 三种 ANN 索引的取舍
- [ ] 能回答「双塔为什么比单塔适合召回」「如何防用户塔过拟合」